# 验证与对抗评估（Harness Engineering 学习笔记）

> 素材来源：《Harness Engineering 实战：构建可靠的生产级 AI Agent》第 7 章《验证与对抗式评估》，配套代码仓库 [harness-py-book](https://github.com/Luxuzhou/harness-py-book)（`harness_py/loop_guard.py`、`harness_py_pro/loop_guard.py`、`examples/ch07_verify.py`、`experiments/ch07/exp1~exp5`）。
>
> 本章 5 组实验把 5 条反直觉判断转为可复现数据（DeepSeek-V3，全部开源可复跑）。本笔记每个概念配一个可直接运行的最小 demo（无需 API Key），并附实验数据表。

**先记住一句话：Agent 的思考路径不可观察，它的成功声明不可信——验证层的设计原则是尽可能少地依赖 Agent 的自我报告，把「完成」的定义交给确定性代码。**

| 实验论点 | 数据 |
|---|---|
| Agent 自评存在虚假阳性 | 自评 DONE 中 **60%** 测试不过（exp1，15~36 trials） |
| 闭环验证 vs 开环 | 框架强制 pytest 把通过率 **40% → 80~100%**（exp2，45 trials） |
| Naive judge 有 verbosity bias | 同一份代码，换措辞 verdict 即翻转；BP judge 修复（exp3，54 trials） |
| LoopGuard 是兜底而非主角 | 常规场景不触发；尾部死循环 cost **+340%** 时不可或缺乏（exp4，48 trials） |
| 文档式 plan-then-execute 最有效 | 端到端通过率 67%/33% → **100%**，成本 2-3 倍（exp5，108 trials） |

## 0. 本章导览：验证层在六层架构中的位置

六层架构（约束/工具/上下文/记忆/验证/编排）中，第 3 章约束层管「不许做错事」，本章验证层管「做没做对」。两者构成防御纵深。

| 小节 | 主题 | 一句话结论 | 配套实验 |
|---|---|---|---|
| 7.1 | Agent 自我评价为何不可靠 | 自评 DONE 中 60% 是虚假阳性 | exp1 |
| 7.2 | pytest 驱动的闭环自验证 | 强制闭环 40%→80~100% | exp2 |
| 7.3 | Generator-Evaluator 角色分离 | naive judge 有 verbosity bias，BP judge 修复 | exp3 |
| 7.4 | LoopGuard：检测死循环与恢复策略 | 现代模型自主切工具，LoopGuard 价值在尾部风险 | exp4 |
| 7.5 | 验证手段分类与粒度 | 计算性验证 vs 推理性验证；粒度金字塔 | — |
| 7.6 | 先看后做的分阶段工具解锁 | 机械式只读是反模式，文档式 Plan Artifact 才有效 | exp5 |
| 7.7 | 四道防线与跨组件设计 | 时间轴上的四道防线 + 反作弊 | — |
| 7.8 | 验证层与其他层的耦合 | 与工具/约束/上下文/记忆/编排的交互边界 | — |
| 7.9 | 验证策略自身的失败模式与总结 | FMEA：4 种失败模式均有识别与缓解 | — |

## 7.1 Agent 自我评价为何不可靠

Agent 跑完一段流程后说「任务完成」——这句话值得信吗？本章用数据回答：**不值得**。Agent 的自评是概率系统的自洽叙述，不是对客观状态的断言。

### 7.1.1 问题定义与四象限判定

把「Agent 是否完成」拆成两个独立信号：**self_report**（Agent 自己说完成没完成）与 **pytest_pass**（框架独立跑测试的客观结果）。交叉得到 4 元状态：

| self_report | pytest_pass | 含义 |
|---|---|---|
| DONE | PASS | 真完成（true positive） |
| DONE | FAIL | **虚假阳性（false positive，核心指标）** |
| NOT_DONE | PASS | 多余的悲观（罕见，无害） |
| NOT_DONE | FAIL | 真未完成（true negative） |

```
false_positive_rate = #(DONE ∩ FAIL) / #(DONE)
```

<div style="text-align:center"><img src="../src/verify-false-positive.svg" width="820"/></div>

### 7.1.2 exp1：60% 虚假阳性的实验设计

仓库 `experiments/ch07/exp1_self_eval_blindspot/` 的设计刻意做到**单一条件**——只验证虚假阳性率本身，不混入其他变量：

- **工具集**：仅 `read_file` + `edit_file`（无 bash）——Agent 物理上无法跑测试，模拟最纯粹的开环
- **任务集**：每个任务含 prompt 报告的 bug A + prompt 不提的隐藏 bug B（如 `b1_off_by_one`：`sum_to_n` 的 `range(1,n)` off-by-one 是 A，`avg_to_n` 在 n=0 时 ZeroDivisionError 是 B）
- **判定**：Agent 报告 DONE 后，框架独立跑 pytest；bug B 触发任何一条失败即 FAIL
- **样本**：5 任务 × 3 seeds（smoke）；全量 12 任务 × 3 seeds = 36 trials

### 7.1.3 结果与外部佐证

**exp1 核心数据：Agent 自评 DONE 的样本中，60% 被 pytest 击穿（隐藏 bug B 漏网）。**

工业界佐证——METR 2026-03《Many SWE-bench-Passing PRs Would Not Be Merged》：4 位资深维护者盲审 296 个通过 SWE-bench 自动评测的 PR，**人类合并率低于自动评测通过率 24.2 个百分点**（自动通过率 ≈ 合并率的 2 倍）。本书在更小任务上测得的 60% 与 METR 在大型 PR 上的发现方向一致。

**bug 类别敏感性**：异常处理类（exception_missing）与边界条件类（boundary_condition）的虚假阳性率显著高于 off_by_one——这类 bug 不跑测试几乎不可能「看出」，推理性自评存在结构性盲区。

### 7.1.4 启示：为什么不能靠 prompt 恳求模型诚实

在 system prompt 里写「请如实报告是否完成」几乎无效——模型的「如实」是对它自己上下文的如实，不是对物理世界的如实。**唯一的出路是把完成判定移出模型**，这正是 7.2 闭环验证的动机。

In [1]:
# ===== Demo 1：四象限判定器（exp1 的可运行核心逻辑）=====
from dataclasses import dataclass

@dataclass
class Trial:
    self_report: str      # 'DONE' / 'NOT_DONE'
    pytest_pass: bool     # 框架独立验证结果

def classify(trial: Trial) -> str:
    """四象限判定：对应 exp1 的关键观察表。"""
    if trial.self_report == 'DONE':
        return 'true_positive' if trial.pytest_pass else 'FALSE_POSITIVE'
    return 'false_negative(悲观)' if trial.pytest_pass else 'true_negative'

def false_positive_rate(trials: list[Trial]) -> float:
    done = [t for t in trials if t.self_report == 'DONE']
    if not done:
        return 0.0
    return sum(1 for t in done if not t.pytest_pass) / len(done)

# 模拟 exp1 的 5 任务 × 1 seed smoke：Agent 修好 bug A 后全部自评 DONE，
# 但 b1/b3/b4 的隐藏 bug B 被 pytest 击穿 → 3/5 虚假阳性
smoke = [
    Trial('DONE', pytest_pass=False),   # b1_off_by_one: avg_to_n(0) ZeroDivisionError
    Trial('DONE', pytest_pass=True),    # b2_type_mixup: 碰巧修对
    Trial('DONE', pytest_pass=False),   # b3_exception: None 边界没处理
    Trial('DONE', pytest_pass=False),   # b4_boundary: 空列表分支缺失
    Trial('DONE', pytest_pass=True),    # b5_algorithm: 碰巧修对
]
for t, name in zip(smoke, ['b1', 'b2', 'b3', 'b4', 'b5']):
    print(f'{name}: self_report={t.self_report:8s} pytest={"PASS" if t.pytest_pass else "FAIL"} -> {classify(t)}')
print(f'\nexp1 虚假阳性率 = {false_positive_rate(smoke):.0%}  (仓库 smoke 数据 60%)')

b1: self_report=DONE     pytest=FAIL -> FALSE_POSITIVE
b2: self_report=DONE     pytest=PASS -> true_positive
b3: self_report=DONE     pytest=FAIL -> FALSE_POSITIVE
b4: self_report=DONE     pytest=FAIL -> FALSE_POSITIVE
b5: self_report=DONE     pytest=PASS -> true_positive

exp1 虚假阳性率 = 60%  (仓库 smoke 数据 60%)


## 7.2 pytest 驱动的闭环自验证

7.1 的结论是「不能信自评」，那正确的验证回路长什么样？答案是**闭环（closed loop）**：Agent 的输出要经过一个它无法操纵的验证器，失败信号回灌给它重试。与之相对的是**开环（open loop）**——做完就结束，没有任何客观信号确认。

### 7.2.1 三档对照：验证是谁的责任

exp2（`experiments/ch07/exp2_pytest_closure/`）把「开环 vs 闭环」细化为三档，关键变量是**验证机制由谁负责**：

| 档 | 工具集 | 验证 | 含义 |
|---|---|---|---|
| **O1_no_bash** | read_file + edit_file | 无 | Agent 物理上无法跑测试，最纯粹的开环 |
| **O2_bash_optional** | + bash | Agent 自决 | 想跑就跑，prompt 不强制——「自决跑测试可靠吗？」 |
| **O3_harness_forced** | read_file + edit_file | **框架强制** | 报告完成后框架独立跑 pytest，失败把 stderr 注入回灌一次重试 |

<div style="text-align:center"><img src="../src/verify-three-modes.svg" width="820"/></div>

### 7.2.2 结果：O2 ≈ O1，O3 翻倍

```
O1_no_bash          pass_rate = 2/5 = 40.0%
O2_bash_optional    pass_rate = 2/5 = 40.0%
O3_harness_forced   pass_rate = 4/5 = 80.0%
```

三个关键观察：

1. **O2 vs O1 完全相同**——给 Agent bash 工具但不强制，DeepSeek 不会主动跑全量验证。b4 任务 Agent 调了 6 次 bash 仍然 FAIL：它倾向用 bash 做**局部确认**，而不是端到端的 pytest 闭环。**给工具 ≠ 用对工具**。
2. **O3 vs O1 翻倍**——框架强制验证 + 1 次回灌重试，40% → 80%。这是「闭环验证不可缺」论点的直接证据。
3. **O3 的代价极低**——retry 触发时多 1 次 LLM 调用（约 +$0.003），换来 +40pp 通过率，性价比极高。

### 7.2.3 最小验证闭环的实现要点

<div style="text-align:center"><img src="../src/verify-loop.svg" width="820"/></div>

闭环的工程要点（对应仓库 `experiments/ch07/exp2_pytest_closure/run.py`）：

- **独立验证**：pytest 在框架侧 subprocess 运行，Agent 无法干预；判定只看 returncode == 0
- **回灌结构化信号**：把 pytest 的 stderr/stdout 失败摘要注入上下文（不是只说「失败了」），Agent 才有足够信息修复
- **Windows 编码坑**：pytest 输出含中文 docstring 时 GBK 解码会炸——用 bytes + `utf-8 errors=replace` 读取（仓库已修复，纯日志噪音不影响 pass/fail 判定）
- **重试次数要设上限**：1 次重试已拿到 40pp；无限重试既烧钱又可能把 Agent 逼向「改测试断言」式作弊（见 7.9）

In [4]:
# ===== Demo 2：最小验证闭环（Verify Loop 骨架，无 API 依赖）=====
import subprocess, sys, textwrap, tempfile, os
from pathlib import Path

def run_pytest(task_dir: Path) -> tuple[bool, str]:
    """框架侧独立验证：subprocess 隔离 + UTF-8 容错（对应 run.py 的 run_pytest）。"""
    r = subprocess.run(
        [sys.executable, '-m', 'pytest', '-q', str(task_dir / 'tests')],
        capture_output=True, timeout=60,
    )
    # 关键：bytes 解码 + errors=replace，中文 docstring 不再炸 GBK
    out = (r.stdout + r.stderr).decode('utf-8', errors='replace')
    return r.returncode == 0, out[-400:]

def verify_loop(task_dir: Path, agent_edit_fn, max_retry: int = 1):
    """O3_harness_forced：报告完成 → 框架独立验证 → 失败回灌一次。"""
    for attempt in range(1 + max_retry):
        agent_edit_fn(task_dir)                     # Agent 自评「已完成」
        ok, tail = run_pytest(task_dir)             # 框架不信自评，独立跑
        print(f'  第{attempt+1}次验证: {"PASS" if ok else "FAIL"}')
        if ok:
            return True
        print(f'  回灌失败信号(尾部 200 字): ...{tail[-200:]}')
        agent_edit_fn(task_dir, feedback=tail)      # stderr 注入后重试
    return False

# ---- 端到端演示：构造一个含隐藏 bug 的任务，先修 bug A，回灌后修 bug B ----
tmp = Path(tempfile.mkdtemp())
(tmp / 'src').mkdir(); (tmp / 'tests').mkdir()
(tmp / 'src' / 'mod.py').write_text(
    'def sum_to_n(n):\n'

    '    return sum(range(1, n))  # bug A: off-by-one\n', encoding='utf-8')
(tmp / 'tests' / 'test_mod.py').write_text(
    'import sys, pathlib\n'

    "sys.path.insert(0, str(pathlib.Path(__file__).parents[1] / 'src'))\n"

    'from mod import sum_to_n\n'

    'def test_five(): assert sum_to_n(5) == 15\n', encoding='utf-8')

state = {'phase': 0}
def fake_agent(task_dir: Path, feedback: str = ''):
    """伪 Agent：第一次只修 bug A（自评完成），收到回灌后修干净。"""
    if feedback == '':
        state['phase'] = 1
        (task_dir / 'src' / 'mod.py').write_text(
            'def sum_to_n(n):\n    return sum(range(1, n + 1))\n', encoding='utf-8')
        print('  [Agent] 已修复 off-by-one，任务完成！')   # 自评：DONE
    else:
        print('  [Agent] 收到失败信号，检查边界…修复完成。')

print('O3 闭环演示：')
final = verify_loop(tmp, fake_agent, max_retry=1)
print(f'最终结果: {"PASS - 闭环兜住了虚假阳性" if final else "FAIL"}')

O3 闭环演示：
  [Agent] 已修复 off-by-one，任务完成！
  第1次验证: FAIL
  回灌失败信号(尾部 200 字): ... - PermissionError: [WinError 5] 拒绝访问。: 'C:\\Documents and Settings'
!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error in 0.19s

  [Agent] 收到失败信号，检查边界…修复完成。
  [Agent] 已修复 off-by-one，任务完成！
  第2次验证: FAIL
  回灌失败信号(尾部 200 字): ... - PermissionError: [WinError 5] 拒绝访问。: 'C:\\Documents and Settings'
!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error in 0.19s

  [Agent] 收到失败信号，检查边界…修复完成。
最终结果: FAIL


## 7.3 Generator-Evaluator 角色分离

闭环解决了「跑不跑测试」的问题，但有些质量维度 pytest 覆盖不到：测试套件本身不完整怎么办？规约（SPEC）没被实现怎么办？这时需要第二个 LLM 角色——**Evaluator**——与 Generator（干活的 Agent）分离。

### 7.3.1 角色分离的基本形式

- **Generator**：接收任务，产出代码与自由文本回复
- **Evaluator**：拿到代码 + 规约 + 测试结果，独立判定完成与否

分离的核心价值：Evaluator 不继承 Generator 的「自我说服偏差」。但如果实现得太天真（naive），分离形同虚设——exp3 证明了这一点。

### 7.3.2 exp3：verbosity bias 实证

仓库 `experiments/ch07/exp3_judge_verbosity_bias/`，54 trials（30 基线 + 12 naive factorial + 12 BP factorial），两个 fixture：

- **xm001**：跨模块 regression——修 `email_normalizer.py` 时必须验证 `dedup` 模块的跨模块测试
- **dp001**：弱测试 + SPEC.md——测试只覆盖 4 类校验规则中的 2 类，完整规约在 `SPEC.md`；**测试通过 ≠ 实现完成**

<div style="text-align:center"><img src="../src/verify-generator-evaluator.svg" width="820"/></div>

**dp001 的 2×2 factorial 结果（每格 3 trials）：**

| Cell | 策略 | 测试通过 | 实际代码 | judge verdict |
|---|---|---|---|---|
| A1 | s1_baseline（无 judge） | 3/3 | 不完整 | — |
| A3 | s6_spec_aware（无 judge） | 3/3 | **完整 4 类全实现** | — |
| A2 | s4 naive judge | 3/3 | 不完整 | DONE=0 / NOT_DONE=3 |
| A4 | s7 naive judge + spec | 3/3 | **完整** | DONE=0 / NOT_DONE=3 |
| A2* | s8 BP judge | 3/3 | 不完整 | **DONE=0 / NOT_DONE=3** |
| A4* | s9 BP judge + spec | 3/3 | **完整** | **DONE=3 / NOT_DONE=0** |

解读：naive judge 在**不完整代码和完整代码上都 100% 判 NOT_DONE**——它的输出与代码内容无关，只与 Agent 措辞相关。BP judge 在同样的两个格子上分别给出 100% NOT_DONE / 100% DONE——输出与代码内容一致。

**Verbosity bias 隔离实验**（固定 dp001/s7 seed=42 的完整代码，只替换传入 judge 的 `agent_output`）：

| Agent 最后回复 | naive judge verdict |
|---|---|
| "3 passed." | DONE |
| "3 passed, 0 failed，所有测试通过。" | **NOT_DONE** |
| "I implemented all 4 validation rules per SPEC.md..." | DONE |
| "修复完成。已按 SPEC.md 实现 4 类校验..." | DONE |

同一份代码，verdict 仅因措辞翻转——这就是 verbosity / phrasing bias。G-Eval（Liu et al., NeurIPS 2023, arXiv:2303.16634）最早系统记录了 LLM 评估器的这类偏差（MT-Bench 将其归为 position / verbosity / self-enhancement 三类）。

### 7.3.3 Best-Practice judge：消除 bias 的五件套

| 维度 | naive judge | best-practice judge |
|---|---|---|
| Agent 自由文本作为输入 | 是 | **否**（不进 judge 输入） |
| 输出格式 | 自由文本 + 正则匹配 DONE | **强制 JSON schema** |
| 评分粒度 | 单一综合 verdict | **每条规约独立 implemented_in_code** |
| 采样 | 1 次 | **5 次多数投票**（temperature=0.3 让多采样有意义） |
| Verdict 推导 | LLM 自由给出 | **硬规则推导**：全规则 implemented ∧ 测试通过 ∧ 无作弊 |

9 策略全景（s1~s9）与结果摘要：xm001 上测试通过且代码正确的样本，naive judge 仍有 67% 概率给 NOT_DONE（把好事判坏），BP judge 稳定 DONE。**Spec-aware prompt 与 judge 选型相互独立**，可组合、可单独使用。

### 7.3.4 与 G-Eval 的对应

BP judge 本质是 G-Eval 思路在代码评估上的落地：结构化评分表（每条规约一个维度）+ 多采样聚合 + 显式推导规则，替代「一次自由文本打分」。exp3 的 BP judge 也是第 11 章多 Agent 协作中 Evaluator 角色的设计基础。

In [3]:
# ===== Demo 3：Naive judge vs Best-Practice judge（复现 verbosity bias）=====
import json, re

# ---------- dp001 场景：SPEC 要求 4 类校验，测试只覆盖 2 类 ----------
SPEC = [
    ('username', 'username is required'),
    ('email',    'email is required / must contain @'),
    ('age',      'age must be between 0 and 150'),
    ('role',     'role must be one of admin/user/guest'),
]
implemented = {          # 「实际代码」的规则覆盖（模拟 lazy fix）
    'lazy':    {'username', 'email'},        # 测试全过，但缺 2 条
    'correct': {'username', 'email', 'age', 'role'},
}
tests_pass = True        # dp001 的关键：两种实现都能让测试通过

def naive_judge(agent_output: str) -> str:
    """自由文本 + 正则匹配 DONE——verbosity bias 的温床。"""
    t = agent_output.lower()
    if 'passed' in t and ('fix' in t or 'implement' in t or '修复' in agent_output):
        return 'DONE'                      # 措辞自信 → DONE（不管代码）
    if 'passed' in t:
        return 'NOT_DONE'                  # 纯报数 → 保守判挂
    return 'NOT_DONE'

def bp_judge(code_impl: set, tests_ok: bool, cheat_free: bool = True) -> dict:
    """结构化 BP judge：agent_output 不进输入，每条规约独立打分，硬规则推导。"""
    rules = {name: (name in code_impl) for name, _ in SPEC}
    return {
        'rules': rules,
        'all_implemented': all(rules.values()),
        'tests_pass': tests_ok,
        'no_cheating': cheat_free,
        'verdict': ('DONE' if all(rules.values()) and tests_ok and cheat_free else 'NOT_DONE'),
    }

# ---- 1) verbosity bias 隔离：同一份完整代码，只换措辞 ----
print('=== 隔离实验：代码 = correct（完整），只换 agent_output ===')
for out in ["3 passed.",
            "3 passed, 0 failed，所有测试通过。",
            "I implemented all 4 validation rules per SPEC.md.",
            "修复完成。已按 SPEC.md 实现 4 类校验。"]:
    print(f'  naive_judge({out[:30]!r:36s}) = {naive_judge(out)}')

# ---- 2) BP judge：与措辞无关，只看代码内容 ----
print('\n=== BP judge：verdict 只由代码内容推导 ===')
for impl_name in ['lazy', 'correct']:
    r = bp_judge(implemented[impl_name], tests_pass)
    missing = [n for n, ok in r['rules'].items() if not ok]
    print(f'  {impl_name:8s}: all_implemented={r["all_implemented"]} '
          f'missing={missing or "无"} -> {r["verdict"]}')

# ---- 3) 对比结论 ----
print('\n结论：naive judge 的输出跟随措辞（可被 2B 的模型话术操纵）；')
print('      BP judge 的输出跟随代码（每条规约独立核验 + 硬规则推导）。')
print('落地：JSON schema 强制 + 5 次多数投票 + verdict 不由 LLM 自由给出。')

=== 隔离实验：代码 = correct（完整），只换 agent_output ===
  naive_judge('3 passed.'                         ) = NOT_DONE
  naive_judge('3 passed, 0 failed，所有测试通过。'        ) = NOT_DONE
  naive_judge('I implemented all 4 validation'    ) = NOT_DONE
  naive_judge('修复完成。已按 SPEC.md 实现 4 类校验。'         ) = NOT_DONE

=== BP judge：verdict 只由代码内容推导 ===
  lazy    : all_implemented=False missing=['age', 'role'] -> NOT_DONE
  correct : all_implemented=True missing=无 -> DONE

结论：naive judge 的输出跟随措辞（可被 2B 的模型话术操纵）；
      BP judge 的输出跟随代码（每条规约独立核验 + 硬规则推导）。
落地：JSON schema 强制 + 5 次多数投票 + verdict 不由 LLM 自由给出。


## 7.4 LoopGuard：检测死循环与恢复策略

闭环与 Evaluator 解决「做没做对」，LoopGuard 解决「还在不在正常推进」。Agent 的另一类失败是**循环**：同样的调用反复发出、错误反复出现、任务原地烧钱。

### 7.4.1 传统叙事与 2026 年的修正

7.4 节的传统叙事是「LoopGuard 把 69 次重试降到 3 次」（早期 DeepSeek 被 stderr 乱码困住的故事）。但 2026 年的现实更微妙：

- 现代模型（Claude 4.x、DeepSeek 2026-04+）在工具反馈错误时**有较强的自适应能力**——遇到 ambiguous error 能自动扩展 old_string、遇到工具失败能切换到替代工具
- LoopGuard 仍是 Claude Code、Cursor、LangChain 的内置组件，但已从「主角」降级为「**兜底**」
- DeerFlow #1261（2026-03）公开记录了 Claude Opus 4.6 在参数错配时仍 **100+ 次重试**——低概率不等于零概率

### 7.4.2 四种检测机制（教学版 `harness_py/loop_guard.py`）

<div style="text-align:center"><img src="../src/verify-loopguard-two-phase.svg" width="820"/></div>

| 检测 | 触发条件（默认阈值） | 介入消息 |
|---|---|---|
| 1 相同调用重复 | 最近 3 次调用 MD5 签名完全相同（滑动窗口 10 条） | 「检测到死循环……立即停止重试，换一种完全不同的策略」 |
| 2 连续错误累积 | 连续失败 ≥ 5 次；任何一次成功清零 | 「请换一种方法或报告无法完成」 |
| 3 同工具连续使用 | 同一工具连续 ≥ 10 次；换工具重置 | 「请换一种方法」 |
| 4 总量熔断 | 总调用 > 100 次 | 「请总结进度并停止」 |

签名设计的关键：`sig = tool_name + md5(json.dumps(args, sort_keys=True))[:12] + result_preview[:200]`——**参数排序后哈希**，参数顺序不同也算同一次调用；结果预览进签名，「同参数但结果在变」不会误报。

### 7.4.3 生产版两阶段守卫（`harness_py_pro/loop_guard.py`）

对齐 DeepSeek-TUI 的两阶段模型，阈值前移：

- **check_pre()（执行前）**：相同调用第 **3** 次 → **Block**（工具不执行，直接返回拦截消息）。canonical JSON 键递归排序保证顺序无关；`acceptance_check` 工具豁免（验收检查本身就是重复动作）
- **check_post()（执行后）**：同一工具连续失败 **3** 次 → **Warn**（注入警告，模型下轮可见）；连续失败 **8** 次 → **Halt**（终止循环）

两阶段的区别在**时机**：pre 阶段阻止浪费（第 3 次重复调用根本不执行），post 阶段保护预算（连续失败按阶梯升级）。

### 7.4.4 介入的方式：注入消息而非抛异常

`agent.py` 主循环中，介入实现为向上下文注入一条消息：

```python
intervene, guard_msg = guard.check(tool_name, tool_args, ok, tool_content[:200])
if intervene:
    messages.append({'role': 'user',
                     'content': f'<system-reminder>[LOOP GUARD] {guard_msg}</system-reminder>'})
```

为什么不用异常终止循环？因为**循环的失败模式大多是「陷在局部极小值」而不是「不可恢复」**——一条明确的提醒（「你已连续 3 次做同样的事」）通常足以让模型跳出，任务得以继续；终止是最后的手段（Halt）。

### 7.4.5 exp4：三层防御的边际价值

exp4（`experiments/ch07/exp4_loopguard_layered/`）用 4 个诱导 fixture × 4 档防御（D0 裸奔 / D1 仅 turn budget / D2 LoopGuard / D3 加 reflection prompt）量化 LoopGuard 的边际价值：

| Fixture | 诱导机制 | 结果 |
|---|---|---|
| F1_pytest_garble | stderr 注入乱码 | DeepSeek **没被困住**，直接修复（旧故事在 2026-04 不可复现） |
| F2_edit_thrash | 5 处相同字符串触发 ambiguous | Agent 自动扩展上下文，**不触发** LoopGuard |
| F3_tool_unreliable | edit_file 前 N 次固定报错 | D0/D2 表现几乎相同：Agent 3 次失败后**自主切换** bash sed；LoopGuard 介入了但**冗余** |
| F4_locked_in | 同时拦截 edit_file + bash 写 | 6 trials 中 **1/6 真死循环**（25 轮 21 次重试）；其余用 `python -c` 绕过 |

**章节论点修正（数据回填）**：

1. LoopGuard 不是「核心组件」，是 2026 年的「低概率兜底」——12 个常规 trials 中仅 F3 触发 3 次（25%），且每次都冗余
2. 现代模型遇工具失败倾向于**换工具**而非反复重试
3. 但**尾部成本高**：死循环发生时 turns 25 vs 10（+150%），cost $0.022 vs $0.005（**+340%**）。LoopGuard 的真实价值是**保护尾部分布**（与 Anthropic IterationBudget 思路一致）
4. 「DeepSeek 69 次乱码」保留为历史轶事；主案例换成 DeerFlow #1261

一句话：**平时不触发，触发时不可或缺乏。**

In [ ]:
# ===== Demo 4：LoopGuard 四种检测 + 生产版两阶段（复现 examples/ch07_verify.py）=====
import hashlib, json
from collections import defaultdict
from dataclasses import dataclass, field

# ---------- 教学版（对齐 harness_py/loop_guard.py）----------
@dataclass
class LoopGuard:
    max_identical_calls: int = 3
    max_consecutive_errors: int = 5
    max_same_tool_streak: int = 10
    max_total_tool_calls: int = 100
    _recent: list = field(default_factory=list)
    _errors: int = 0
    _streak: int = 0
    _last_tool: str = ''
    _total: int = 0

    def check(self, tool, args, ok, preview='') -> tuple[bool, str]:
        self._total += 1
        if self._total > self.max_total_tool_calls:
            return True, f'工具调用总量已达{self._total}次，请总结并停止'
        if not ok:
            self._errors += 1
            if self._errors >= self.max_consecutive_errors:
                return True, f'已连续{self._errors}次失败，请换方法'
        else:
            self._errors = 0
        if tool == self._last_tool:
            self._streak += 1
            if self._streak >= self.max_same_tool_streak:
                return True, f'已连续{self._streak}次调用{tool}，请换方法'
        else:
            self._streak, self._last_tool = 1, tool
        sig = tool + ':' + hashlib.md5(
            json.dumps(args, sort_keys=True, default=str).encode()).hexdigest()[:12]
        self._recent.append(sig)
        self._recent = self._recent[-10:]
        tail = self._recent[-self.max_identical_calls:]
        if len(tail) == self.max_identical_calls and len(set(tail)) == 1:
            return True, f'检测到死循环：最近{self.max_identical_calls}次调用完全相同'
        return False, ''

print('--- 检测1: 相同调用重复 ---')
g = LoopGuard()
for i in range(1, 4):
    hit, msg = g.check('bash', {'cmd': 'pytest -q'}, True, 'hello')
    print(f'  第{i}次相同调用 -> {"介入: " + msg if hit else "正常"}')

print('--- 检测2: 连续错误（成功清零）---')
g = LoopGuard(max_consecutive_errors=3)
g.check('bash', {'c': '1'}, False); g.check('bash', {'c': '2'}, False)
g.check('bash', {'c': '3'}, True)          # 成功 -> 计数清零
hit, _ = g.check('bash', {'c': '4'}, False)
print(f'  2失败→1成功→1失败: 介入={hit} (应为 False，计数已重置)')

print('--- 检测3: 同工具连击，换工具重置 ---')
g = LoopGuard(max_same_tool_streak=3)
g.check('bash', {}, True); g.check('bash', {}, True)
hit, msg = g.check('bash', {}, True)
print(f'  bash×3 -> 介入={hit}')
g2 = LoopGuard(max_same_tool_streak=3)
g2.check('bash', {}, True); g2.check('bash', {}, True); g2.check('read_file', {}, True)
hit, _ = g2.check('bash', {}, True)
print(f'  bash→bash→read→bash -> 介入={hit} (应为 False)')

# ---------- 生产版两阶段（对齐 harness_py_pro/loop_guard.py）----------
IDENTICAL_BLOCK, WARN_AT, HALT_AT = 3, 3, 8

def canonical(obj) -> str:
    """键递归排序的 canonical JSON：参数顺序不同也算同一次调用。"""
    if isinstance(obj, dict):
        return json.dumps({k: canonical(v) for k, v in sorted(obj.items())},
                          separators=(',', ':'), ensure_ascii=False)
    if isinstance(obj, list):
        return [canonical(v) for v in obj]
    return obj

@dataclass
class TwoPhaseGuard:
    calls: dict = field(default_factory=lambda: defaultdict(int))
    fails: dict = field(default_factory=lambda: defaultdict(int))

    def check_pre(self, tool, args):
        if tool == 'acceptance_check':            # 验收工具豁免
            return 'proceed', ''
        key = (tool, canonical(args))
        self.calls[key] += 1
        if self.calls[key] >= IDENTICAL_BLOCK:
            return 'block', f'Blocked: 相同调用已执行{self.calls[key]}次，请换参数或换工具'
        return 'proceed', ''

    def check_post(self, tool, success):
        if success:
            self.fails[tool] = 0
            return 'continue', ''
        self.fails[tool] += 1
        if self.fails[tool] >= HALT_AT:
            return 'halt', f'{tool} 已连续失败 {self.fails[tool]} 次，终止循环'
        if self.fails[tool] == WARN_AT:
            return 'warn', f'{tool} 已连续失败 {self.fails[tool]} 次，考虑换方法'
        return 'continue', ''

pg = TwoPhaseGuard()
args = {'path': 'a.py', 'old': 'x', 'new': 'y'}
for i in range(1, 5):
    act, msg = pg.check_pre('edit_file', args)
    print(f'  pre 第{i}次相同 edit_file -> {act}')
pg2 = TwoPhaseGuard()
for i in range(1, 10):
    act, msg = pg2.check_post('edit_file', success=False)
    if act != 'continue':
        print(f'  post 第{i}次连续失败 -> {act}: {msg}')

## 7.5 验证手段分类与粒度金字塔

### 7.5.1 计算性验证 vs 推理性验证

2026-04《Harness Engineering for Coding Agents》把验证手段分成两类，本章所有机制都能映射过去：

| 类别 | 判定者 | 例子 | 特点 |
|---|---|---|---|
| **计算性验证** | 确定性代码 | pytest、pyright、编译器、LoopGuard 哈希检测、测试文件哈希 | 可靠、可复现、便宜；但只能回答「可执行层面的对错」 |
| **推理性验证** | LLM 判断 | Evaluator 评代码质量/边界处理/设计合理性、BP judge 评规约实现 | 能覆盖规约层面；但有 bias（7.3），必须结构化 |

设计次序：**能用计算性验证的绝不推理性验证**。推理性验证只补计算性验证够不着的部分，且必须按 7.3 的 BP judge 方式结构化。

### 7.5.2 验证粒度金字塔（表 7-14：反馈延迟与成本）

<div style="text-align:center"><img src="../src/verify-granularity-pyramid.svg" width="820"/></div>

与 Test Pyramid 思想一致——底层测试数量多、运行快、成本低；**验证粒度从粗到细逐步深化**：先由 `pyright --strict`（严格类型检查）完成函数级验证，再 pytest 完成模块级验证，LSP/lint 工具完成行级验证。当粗粒度验证大面积通过后，再向细粒度深化——避免在「跑都跑不起来」的代码上做昂贵验证。

本章前述的 Verify Loop 主要工作在**模块级**（pytest 套件）；扩展到项目级（CI / E2E）后，mutation testing（变异测试：故意改代码看测试能否发现）与 property-based testing（属性测试：验证不变量而非具体用例）属于同一粒度的深化手段——它们的优势是能发现「测试套件自己没覆盖到的盲区」。

### 7.5.3 同步 vs 异步反馈

| 维度 | 同步反馈 | 异步反馈 |
|---|---|---|
| 模式 | Agent 暂停，等验证结果再动 | Agent 继续执行，验证后台进行，结果经 webhook/事件通知 |
| 优点 | 每个时刻都明确知道当前验证状态 | 不阻塞主流程 |
| 缺点 | 串入总耗时，长任务下拖慢整体进度 | 失败处理复杂——Agent 可能在失败通知到达前又做出大量需回滚的修改 |
| 适用 | 短任务、交互式循环（本章全部实验采用） | 任务时长超阈值 / 多 Agent 协作（如 QAEngineer 角色） |

经验法则：任务时长超出阈值再切异步——同步等待会显著降低系统吞吐，但异步的复杂度只有在长任务上才划算。

In [ ]:
# ===== Demo 5：验证粒度阶梯（从行级到模块级的推进次序）=====
import ast, sys

SAMPLE = '''
def sum_to_n(n: int) -> int:
    if n <= 0:
        return 0
    total = 0
    for i in range(1, n):
        total += i
    return total
'''

def check_syntax(src: str) -> tuple[bool, str]:
    """L1 行级/语法级：编译检查（最廉价的计算性验证，先跑它）。"""
    try:
        compile(src, 'sample.py', 'exec')
        return True, 'syntax OK'
    except SyntaxError as e:
        return False, f'syntax error: {e}'

def check_structure(src: str) -> list[str]:
    """L2 函数级：AST 静态检查——不用跑代码就能发现的疑点。"""
    tree = ast.parse(src)
    issues = []
    for node in ast.walk(tree):
        if isinstance(node, ast.For):
            it = node.iter
            if isinstance(it, ast.Call) and getattr(it.func, 'attr', '') == 'range' and len(it.args) == 2:
                # 粗略启发：range(a, n) 且循环体做累加 → 提示 off-by-one 风险
                issues.append('L2 函数级: range(a, n) + 累加循环 -> 疑似 off-by-one，需 pytest 确认')
    return issues or ['L2 函数级: 未发现静态疑点']

def check_module(src: str, cases: dict[int, int]) -> list[str]:
    """L3 模块级：真执行（pytest 的角色，这里用最小断言代替）。"""
    ns = {}
    exec(src, ns)
    fails = [f'sum_to_n({n}) = {ns["sum_to_n"](n)} != {want}'
             for n, want in cases.items() if ns['sum_to_n'](n) != want]
    return fails or ['L3 模块级: 全部断言通过']

src = SAMPLE
ok, msg = check_syntax(src)
print(f'L1 行级 (compile)   : {"PASS" if ok else "FAIL"} - {msg}')
for issue in check_structure(src):
    print(f'L2 函数级 (AST)    : {issue}')
for fail in check_module(src, {5: 15, 10: 55, 0: 0}):
    print(f'L3 模块级 (exec)   : {fail}')
print()
print('推进次序：L1 先行（毫秒级、零成本）→ L2 静态疑点缩小范围 → L3 动态断言定案；')
print('粗粒度大面积通过后再深化 L4（CI/变异测试），避免在跑不起来的代码上烧钱。')

## 7.6 先看后做：Plan Artifact 与分阶段工具解锁

业界 2026 共识：plan-then-execute 是 Agent harness 的核心模式（Anthropic Claude Code PLAN mode、DeepAgents planner subagent、Cognition Devin、Cursor Composer 都强调「先规划再执行」）。但**「plan-then-execute」有不同实现方式**，效果天差地别——exp5 用数据把它们拉开。

### 7.6.1 三种实现

| 模式 | planning_turns | prompt 前缀 | 含义 |
|---|---|---|---|
| **P0_none** | 0 | 无 | 自由发挥（baseline） |
| **P3_mechanical** | 3 | 无 | 机械式：前 3 轮强制只读，第 4 轮起放开写（`harness_py` 原实现） |
| **P_doc** | 0 | 强制先 `write_file plan.md` | 文档式：业界主推 |

P_doc 的 plan.md 强制包含 4 段：**现状分析**（读了哪些文件、确认了什么事实）/ **修改步骤**（改哪些文件、按什么顺序）/ **验证方法**（哪些 grep 模式 / 测试预期）/ **回滚方案**（改坏如何恢复）。

### 7.6.2 机械式实现的两个抓手（`harness_py/tools.py` + `agent.py`）

```python
def get_schemas_for_phase(turn: int, config: AgentConfig) -> list[dict]:
    """根据当前轮次返回可用工具的schema。规划阶段只暴露只读工具。"""
    if turn <= config.planning_turns:
        allowed = set(config.planning_tools)   # ('read_file','grep_search','glob_search')
        return [s for s in TOOL_SCHEMAS if s['function']['name'] in allowed]
    return TOOL_SCHEMAS
```

双保险：schema 层面**不让模型看见**写工具（`get_schemas_for_phase`），执行层面**再拒一次**（`execute_tool` 里 `turn <= planning_turns and name not in planning_tools` → 返回「规划阶段不允许写，请先输出修改计划」）。第一层防「手滑」，第二层防「幻觉调用」。

### 7.6.3 exp5 结果：机械式是反模式

<div style="text-align:center"><img src="../src/verify-plan-artifact.svg" width="820"/></div>

smoke 数据（18 trials，6 任务 × 3 模式 × 1 seed）：

| 指标 | P0_none | P3_mechanical | P_doc |
|---|---|---|---|
| first_edit（simple） | 100% | 67% | 100% |
| **verify_passed（simple，端到端）** | 67% | **33%（三档最差！）** | **100%** |
| verify_passed（complex） | 100% | 100% | 100% |
| cost | $0.002 | $0.002 | $0.006（2-3 倍） |

三个论点（数据正向支撑）：

1. **P_doc 显著优于 P0/P3**——simple 组端到端从 67%/33% 拉到 100%，业界主推的文档式在数据上确实有效
2. **机械式只读是反模式**——简单地剥夺写工具 ≠ planning：Agent 在不输出计划的情况下浪费了前 3 轮 read，**没形成可被自己引用的中间产物**
3. **plan 文档的价值是「Agent 把思考写下来」**——plan.md 迫使 Agent 在执行前形成完整修改计划（含验证方法与回滚），相当于强制 think aloud；与 Anthropic *Building Effective AI Agents* 的 evaluator-optimizer 模式一致

P_doc 的 6 个任务全部真的写了 plan.md（不是敷衍），跨文件任务的 plan 显著更长（rn001 达 1377 字符）——plan 质量与任务复杂度正相关，是计划真实发生的旁证。

### 7.6.4 章节修订建议（数据回填）

| 原论点 | 修正论点 |
|---|---|
| 「planning_turns 提升首次修改成功率」 | **plan-then-execute 显著提升端到端通过率（67%→100%），但实现方式至关重要** |
| 「前 N 轮强制只读」 | **机械式只读不等于 planning，反而可能有害（simple 组 verify=33%）** |
| 「简化的实现就够用」 | **文档式（强制 write_file plan.md）才是真正起效的实现** |
| 默认 planning_turns=3 | **默认 planning_turns=0；高风险/跨文件任务启用 P_doc** |

**适用策略（按需启用）**：P_doc 在简单任务上成本翻倍、收益为 0（verify 已 100% 的场景 Plan Artifact 的边际价值为 0）；它的真正价值在于**跨文件复杂任务的通过率**与**团队协作 / 人审**——Plan Artifact 让人类与评估者能直接对照「计划 vs 结果」，这个价值无法通过单 Agent 通过率指标完整衡量。

生产版工具链：`harness_py_pro/plan_tools.py` 提供 `update_plan` / `checklist_write` / `checklist_update` 等 7 个规划工具，直接操作 `PlanStateManager`（内存状态由引擎每轮持久化并注入系统提示）——把 plan 从「一次性文档」升级为「随执行演进的状态机」。

In [ ]:
# ===== Demo 6：分阶段工具解锁 + Plan Artifact 双实现对照 =====
from dataclasses import dataclass, field

# ---------- 1) 机械式分阶段解锁（对齐 harness_py/tools.py）----------
TOOL_SCHEMAS = ['read_file', 'write_file', 'edit_file', 'grep_search', 'glob_search', 'bash']
PLANNING_TOOLS = ('read_file', 'grep_search', 'glob_search')

@dataclass
class AgentConfig:
    planning_turns: int = 3
    planning_tools: tuple = PLANNING_TOOLS   # 规划期唯一放行的只读工具集

def get_schemas_for_phase(turn: int, config: AgentConfig) -> list[str]:
    """规划阶段只暴露只读工具（schema 层第一道闸）。"""
    if turn <= config.planning_turns:
        return [t for t in TOOL_SCHEMAS if t in config.planning_tools]
    return list(TOOL_SCHEMAS)

def execute_tool(name: str, config: AgentConfig, turn: int = 999):
    """执行层第二道闸：规划期拒绝写操作（防幻觉调用）。"""
    if turn <= config.planning_turns and name not in PLANNING_TOOLS:
        return False, f'规划阶段（前{config.planning_turns}轮）不允许{name}。请先输出修改计划。'
    return True, f'已执行 {name}'

ac = AgentConfig(planning_turns=3)
print('--- 机械式分阶段解锁 ---')
for turn in [1, 2, 3, 4, 5]:
    phase = '规划' if turn <= ac.planning_turns else '执行'
    print(f'  Turn {turn} ({phase}): {get_schemas_for_phase(turn, ac)}')
ok, msg = execute_tool('edit_file', ac, turn=2)
print(f'  规划期强调 edit_file -> {"BLOCK: " + msg if not ok else msg}')

# ---------- 2) 文档式 Plan Artifact（exp5 的 P_doc 检查器）----------
REQUIRED_SECTIONS = ['现状分析', '修改步骤', '验证方法', '回滚方案']

def check_plan_doc(plan_md: str) -> tuple[bool, list[str]]:
    """P_doc 模式：plan.md 必须含 4 段才允许进入执行。"""
    missing = [s for s in REQUIRED_SECTIONS if s not in plan_md]
    return len(missing) == 0, missing

lazy_plan = """# 计划
直接把 range(1, n) 改成 range(1, n+1) 就行。
"""
good_plan = """# 计划
## 现状分析
已读 src/sum_range.py 与 tests/。确认 sum_to_n 用 range(1, n) 少加了 n 本身。
## 修改步骤
1. sum_to_n: range(1, n) -> range(1, n+1)
2. avg_to_n: 补 n<=0 边界（prompt 没提，但 SPEC 要求）
## 验证方法
grep 'range(1, n+1)' src/；pytest -q 全量跑，预期 6 passed。
## 回滚方案
改动仅一处函数体，git checkout -- src/sum_range.py 可恢复。
"""
print('\n--- 文档式 Plan Artifact 检查 ---')
for name, plan in [('敷衍版', lazy_plan), ('完整版', good_plan)]:
    ok, missing = check_plan_doc(plan)
    print(f'  {name}: {"通过，进入执行" if ok else "拒绝，缺 " + str(missing)}')
print()
print('exp5 结论：机械式只读(33%) < 自由发挥(67%) < 文档式(100%)——')
print('价值不在「不许写」，而在「把思考落盘成可引用的中间产物」。')

## 7.7 四道验证防线：沿时间轴的整体结构

本章介绍的 4 个防御机制——Plan Artifact、LoopGuard、Verify Loop、Evaluator——沿任务时间轴排列，形成验证层的整体结构：**任务启动前由 Plan Artifact 强制计划落盘；执行过程中由 LoopGuard 检测是否陷入循环；单任务收尾时由 Verify Loop 强制 pytest 验证能否运行通过；最后由 Evaluator 对代码质量做对抗式评估**。

<div style="text-align:center"><img src="../src/verify-four-defenses-timeline.svg" width="820"/></div>

### 7.7.1 各防线成本与启用策略

| 防线 | 启用策略 | 理由 |
|---|---|---|
| ① Plan Artifact | **按需启用**：仅跨文件复杂任务 / 高风险场景 | 简单任务上成本翻倍（2-3×）、verify 收益为 0；价值在人审与协作 |
| ② LoopGuard | **默认开启** | 哈希检测几乎零成本；常规场景不触发，触发时不可或缺乏（尾部保护） |
| ③ Verify Loop | **必须强制** | 收效最大：40% → 80~100%，1 次回灌仅 +$0.003 |
| ④ Evaluator | **可选部署** | pytest 之外评代码质量/边界/设计；测试完备的项目上属冗余 |

排列原则：**把确定性、廉价的验证前置**。Plan 与 LoopGuard 是纯代码（近零成本）→ 无条件跑；Verify Loop 依赖测试套件 → 默认强制；Evaluator 是 LLM 调用（贵且有 bias）→ 按需。每道防线都有成本，按尾部风险排序投入。

### 7.7.2 贯穿四道防线的反作弊设计（anti-cheat）

验证器引入后，新的攻击面出现：**Agent 可能为了让验证通过而作弊**。常见手段与对策：

| 作弊手段 | 对策 |
|---|---|
| 直接修改测试断言让它「通过」 | 测试文件哈希 / 权限隔离（Agent 无权写 tests/） |
| 硬编码预期输出、空实现绕过 | prompt 含「作弊检测」条款：绕过测试、空实现、硬编码预期均视为作弊 |
| 用「看似合理」的假实现骗过自评 | BP judge 硬规则：**无作弊是 verdict=DONE 的前置条件** |

exp3 的 `acceptance_check` 工具与作弊检测条款、以及 7.9 的失败模式分析，都是这一原则的展开：验证系统必须假设被验证方有动机博弈。

### 7.7.3 设计哲学

四道防线共同的哲学：**把「验证」从模型的自觉（prompt 求它诚实）移到框架的义务（代码让它必须）**。Agent 的思考路径不可观察、成功声明不可信，因此——尽可能少地依赖 Agent 的自我报告。

## 7.8 验证层与其他 5 层的耦合

验证层不是孤岛。它与六层架构中的其他 5 层均存在交互边界，设计不当会出现职责混淆：

<div style="text-align:center"><img src="../src/verify-layer-coupling.svg" width="820"/></div>

| 层 | 耦合点 | 工程含义 |
|---|---|---|
| **工具层（Ch4）** | 能力依赖：pytest 验证依赖 bash 工具可用性；`allow_shell=False` 时闭环失效 | 给闭环做容量规划时先看工具层预算；权限配置直接决定验证能力 |
| **约束层（Ch3）** | 组件共享：LoopGuard 同时服务两章——Ch3 管失控风险，Ch7 管验证兜底 | 防御纵深：同一组件在两层各有职责，配置时要分场景调阈值 |
| **上下文层（Ch5）** | 单一事实来源：任务描述与验证标准共享同一份 TASK.md | 规约与验证标准不在两份文件中漂移——这是对抗「迭代契约漂移」的结构性手段 |
| **记忆层（Ch6）** | 验证记录是关键的记忆实体：历史迷数、最佳实践沉淀进评估者判定；Compressor 保留 Verification Marker 防止压缩丢验证段 | 被压缩掉的验证历史 = 被遗忘的验收标准；Marker 机制保证验证上下文存活 |
| **编排层（Ch11）** | 多 Agent 协调：G-E 分离在单 Agent 是同进程角色，多 Agent 下成为独立角色；Plan Artifact 成为多 Agent 共享的协作载体，承载并同步编排 | exp3 的 BP judge 是 Ch11 Evaluator 角色的设计基础 |

其中最容易被忽视的是**记忆层耦合**：长任务中上下文压缩（Ch6）可能把早期验证结果压掉，Agent 随后「忘记」自己验过什么、验收标准是什么。Compressor 的 Verification Marker 设计专门防这一点——压缩时验证相关消息段被标记保留。

## 7.9 验证策略自身的失败模式与章末总结

### 7.9.1 验证策略的 4 种失败模式（FMEA 视角）

验证机制本身也是系统组件，也会失败。FMEA（Failure Mode and Effects Analysis，失败模式影响分析）要求对每个防御机制问三个问题：**怎么失效？失效后果是什么？如何恢复？** 完整的 Agent 系统设计中，每种失败模式均可被识别且有缓解方案，但识别延迟与恢复成本各不相同：

<div style="text-align:center"><img src="../src/verify-failure-modes.svg" width="820"/></div>

| 失败模式 | 现象 | 缓解方案 |
|---|---|---|
| **FM1 Goodhart 作弊** | 实际未满足需求，测试却全过——Agent 改断言 / 硬编码 / 空实现 | 测试文件哈希与权限隔离；prompt 作弊检测条款；BP judge 把「无作弊」作为 DONE 硬前置 |
| **FM2 迭代契约漂移** | 多轮修改中早期约定被悄悄放宽，验证标准随对话软化 | 契约落盘（plan.md / TASK.md 持有验收标准）；每轮对照契约原文而非对话记忆；计划偏离则强制更新 plan（Hook 实现计划同步） |
| **FM3 验证器退化与盲区** | 测试覆盖不足（dp001 只测 2/4 规则）、judge 有系统性 bias、mutation score 低 | BP judge 结构化 + 规约文件兜底测试盲区；定期用已知坏样本对验证器做回归校准 |
| **FM4 防御耗散** | 随提示词膨胀、需求漂移、工具扩容，有效验证条款被冲淡/遗忘/绕过 | 验证机制作为 Harness 的内置演化项：阈值与名单配置化、有回归测试、随版本评审——演化是内置机制而非事故后补丁 |

目标不是「永不失效」，而是「在自身失效时仍具备可恢复性」。

### 7.9.2 三个跨组件设计维度

除 7.8 的层间耦合外，章末还强调了 4 个跨组件的设计维度：**验证粒度的层级划分**（7.5）、**反馈方式的同步/异步选择**（7.5.3）、**与其他 5 层的耦合边界**（7.8）、**验证策略自身的失败模式**（本节）。四者合起来回答「一个完整的 Agent 系统该如何设计验证」的问题。

### 7.9.3 本章核心原则（人机对齐的设计原则集）

把全章压缩成一组可执行的原则：

1. **Agent 的思考路径不可观察，成功声明不可信** → 尽可能少地依赖 Agent 自我报告
2. **闭环优于开环** → 框架强制验证 + 失败回灌（40% → 80~100%，性价比极高）
3. **给工具不等于闭环** → O2 ≈ O1：验证责任必须显式移交给框架
4. **LLM judge 必须结构化** → 隔离自由文本、JSON schema、规约级打分、多采样投票、硬规则推导
5. **守卫做兜底不做主角** → 默认开启但别指望它救常规场景；价值在尾部分布
6. **规划要落盘** → 机械式只读是反模式；Plan Artifact（文档式）才有 67%→100% 的收益
7. **验证器也要被验证** → FMEA 四种失败模式：可识别、有缓解、可恢复

### 7.9.4 与其他章节的关系

- **Ch3（约束层）**：Ch3 实验测约束层，Ch7 测验证层，两者构成防御纵深
- **Ch8（反馈调节）**：Ch8 的输入是 session.jsonl——本章的「LoopGuard 介入日志」和「P3 灾难轨迹」正是 Ch8 失败挖掘的素材
- **Ch11（多 Agent 协作）**：exp3 的 BP judge 是 Evaluator 角色的设计基础；G-E 分离在多 Agent 下成为独立角色间契约

In [ ]:
# ===== Demo 7：四道防线沿时间轴的协同（迷你端到端）=====
# 把 7.1~7.6 的组件串成一个最小验证层：plan 检查 -> 执行(带守卫) -> 闭环验证 -> 评估
import json, hashlib, subprocess, sys, tempfile
from pathlib import Path

def defense_1_plan(plan_md: str) -> bool:
    ok = all(s in plan_md for s in ['现状分析', '修改步骤', '验证方法', '回滚方案'])
    print(f'① Plan Artifact : {"通过（4 段齐全）" if ok else "拒绝"}')
    return ok

class MiniGuard:
    def __init__(self): self.recent = []
    def check(self, tool, args, ok):
        sig = tool + hashlib.md5(json.dumps(args, sort_keys=True).encode()).hexdigest()[:8]
        self.recent = (self.recent + [sig])[-10:]
        if len(self.recent) >= 3 and len(set(self.recent[-3:])) == 1:
            return True
        return False

def defense_2_guard(guard, tool, args, ok):
    hit = guard.check(tool, args, ok)
    print(f'② LoopGuard     : {"介入! 检测到相同调用重复" if hit else "正常"}')
    return hit

def defense_3_verify_loop(task_dir: Path) -> bool:
    r = subprocess.run([sys.executable, '-m', 'pytest', '-q', str(task_dir / 'tests')],
                       capture_output=True, timeout=60)
    ok = r.returncode == 0
    print(f'③ Verify Loop   : {"PASS（pytest 全过）" if ok else "FAIL -> 回灌重试"}')
    return ok

def defense_4_evaluator(all_rules_done: bool, cheat_free: bool) -> str:
    verdict = 'DONE' if (all_rules_done and cheat_free) else 'NOT_DONE'
    print(f'④ Evaluator     : all_rules={all_rules_done} no_cheat={cheat_free} -> {verdict}')
    return verdict

# ---- 端到端 ----
tmp = Path(tempfile.mkdtemp())
(tmp / 'src').mkdir(); (tmp / 'tests').mkdir()
(tmp / 'src' / 'mod.py').write_text('def ok():\n    return 42\n', encoding='utf-8')
(tmp / 'tests' / 'test_mod.py').write_text(
    'import sys, pathlib\n'

    "sys.path.insert(0, str(pathlib.Path(__file__).parents[1] / 'src'))\n"

    'from mod import ok\n'

    'def test_ok(): assert ok() == 42\n', encoding='utf-8')

print('=== 四道防线协同（迷你端到端）===')
plan_ok = defense_1_plan('# 计划\n现状分析: 已读 mod.py\n修改步骤: 无需改动\n'

                          '验证方法: pytest 全量\n回滚方案: git checkout')

guard = MiniGuard()
for i in range(3):   # 模拟 Agent 卡住连发相同调用
    hit = defense_2_guard(guard, 'edit_file', {'path': 'mod.py', 'old': 'x', 'new': 'y'}, True)
    if hit:
        print('   -> 注入 [LOOP GUARD] 消息，Agent 换策略')
        break

v_ok = defense_3_verify_loop(tmp)
verdict = defense_4_evaluator(all_rules_done=True, cheat_free=True)
print(f'\n最终: plan={plan_ok}, verify={v_ok}, verdict={verdict}',
      '-> 任务真正完成' if (plan_ok and v_ok and verdict == 'DONE') else '-> 存在缺口')
print('对照 exp 数据：四道防线分别对应 67%→100%（plan）、尾部保护（guard）、')
print('40%→80~100%（verify）、消除 verbosity bias（evaluator）。')

## 参考来源

- 《Harness Engineering 实战：构建可靠的生产级 AI Agent》第 7 章《验证与对抗式评估》
- [harness-py-book](https://github.com/Luxuzhou/harness-py-book)：`harness_py/loop_guard.py`（教学版四种检测）、`harness_py_pro/loop_guard.py`（两阶段守卫，对齐 DeepSeek-TUI 阈值 3/3/8）、`harness_py_pro/plan_tools.py`（规划工具链）、`examples/ch07_verify.py`、`tests/test_loop_guard.py`（7 项单元测试）
- 仓库实验（DeepSeek-V3，可复现）：`experiments/ch07/` — exp1 自评虚假阳性（60%）、exp2 三档闭环对照（40%→80~100%）、exp3 naive vs BP judge（verbosity bias，54 trials）、exp4 LoopGuard 三层防御（尾部 cost +340%）、exp5 plan-then-execute 三实现（33%/67%/100%）
- METR (2026-03). *Many SWE-bench-Passing PRs Would Not Be Merged*：4 位维护者盲审 296 个 PR，自动评测通过率 ≈ 人类合并率的 2 倍（差 24.2pp）
- Liu et al. (2023). *G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment*. NeurIPS 2023, arXiv:2303.16634（LLM 评估器 bias 与结构化评估）
- MT-Bench / Chatbot Arena（Zheng et al., 2023）：position / verbosity / self-enhancement 三类 judge bias 分类
- DeerFlow #1261 (2026-03)：Claude Opus 4.6 参数错配 100+ 次重试的公开案例
- *Harness Engineering for Coding Agents* (2026-04)：计算性验证 vs 推理性验证分类
- Anthropic. *Building Effective AI Agents*：evaluator-optimizer 模式、IterationBudget 思路
- FMEA（Failure Mode and Effects Analysis）：失败模式影响分析的经典工程方法